# Notebook 4: Ensembles and Uncertainty (Probabilistic Forecasting)

## Theoretical Foundation
As per **Section 2.6 (Combining forecasts)**, ensembles reduce variance and consistently improve accuracy. 
Additionally, **Section 2.3.21 (Estimation and representation of uncertainty)** dictates that point forecasts are insufficient; we must provide prediction intervals.

Here we:
1. Train a secondary model (CatBoost) to ensemble with our LightGBM model.
2. Generate probabilistic predictions using Quantile Regression.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('data/02_hierarchical_features.parquet')
df['month'] = pd.to_datetime(df['month'])
FEATURE_COLS = [c for c in df.columns if c.startswith(('lag', 'roll', 'month_', 'quarter'))]
TARGET = 'demand_transformed'

# We use the final 6 months as our actual Test set
test_start = sorted(df['month'].unique())[-6]
train_df = df[df['month'] < test_start]
test_df = df[df['month'] >= test_start].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_test, y_test = test_df[FEATURE_COLS], test_df[TARGET]

### 1. Ensembling
Train LightGBM and CatBoost, then average their predictions.

In [ ]:
# LightGBM (Point Forecast)
lgb_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_preds = lgb_model.predict(X_test)

# CatBoost (Point Forecast)
cb_model = CatBoostRegressor(iterations=200, learning_rate=0.05, random_seed=42, verbose=0)
cb_model.fit(X_train, y_train)
cb_preds = cb_model.predict(X_test)

# Simple Average Ensemble
ensemble_preds = (lgb_preds + cb_preds) / 2.0
test_df['pred_mean'] = ensemble_preds

### 2. Probabilistic Forecasting (Prediction Intervals)
Train quantile models for the 10th and 90th percentiles to provide an 80% Prediction Interval.

In [ ]:
# LightGBM Quantile Regression
lgb_q10 = lgb.LGBMRegressor(objective='quantile', alpha=0.1, n_estimators=200, verbose=-1)
lgb_q10.fit(X_train, y_train)
test_df['pred_q10'] = lgb_q10.predict(X_test)

lgb_q90 = lgb.LGBMRegressor(objective='quantile', alpha=0.9, n_estimators=200, verbose=-1)
lgb_q90.fit(X_train, y_train)
test_df['pred_q90'] = lgb_q90.predict(X_test)

test_df.to_parquet('data/04_ensemble_probabilistic_preds.parquet', index=False)
print("Ensemble and probabilistic predictions saved.")